# 📄 Day 6A — Build the Store
**PDF → Split → Embed → Save**

---

By the end of this notebook you will have:
- Loaded text from a real tax PDF
- Split it into 400-word chunks
- Turned each chunk into a vector (embedding)
- Saved everything to `store.json` so Notebook 6B can use it

**Notebook 6B** is where you ask questions and get cited answers.

---
## ⏱️ Time: 45 minutes

## Step 1 — Install and Import

In [17]:
%pip install openai pymupdf numpy --quiet


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [18]:
import json
import numpy as np
import fitz                   # PyMuPDF — reads PDFs
from getpass import getpass
from openai import AzureOpenAI

print("✅ Ready")

✅ Ready


## Step 2 — Connect to Azure OpenAI

In [19]:
AZURE_OPENAI_ENDPOINT      = input("Endpoint (e.g. https://xxx.openai.azure.com/): ").strip()
AZURE_OPENAI_API_KEY       = getpass("API Key: ")
AZURE_DEPLOYMENT_NAME      = input("Chat deployment (e.g. gpt-4o): ").strip()
AZURE_EMBEDDING_DEPLOYMENT = input("Embeddings deployment (e.g. text-embedding-ada-002): ").strip()
AZURE_API_VERSION          = "2024-08-01-preview"

client = AzureOpenAI(
    azure_endpoint = AZURE_OPENAI_ENDPOINT,
    api_key        = AZURE_OPENAI_API_KEY,
    api_version    = AZURE_API_VERSION,
)
print("✅ Azure OpenAI client initialised successfully!")

Endpoint (e.g. https://xxx.openai.azure.com/):  https://agentic-ai-tt.openai.azure.com/
API Key:  ········
Chat deployment (e.g. gpt-4o):  gpt-4o
Embeddings deployment (e.g. text-embedding-ada-002):  text-embedding-ada-002


✅ Azure OpenAI client initialised successfully!


## Step 3 — Load the PDF

Set `PDF_PATH` to your file. If you don't have one, leave it as `None` — a demo circular will be used instead.

In [20]:
PDF_PATH = None    # e.g. "gst_circular_01_2023.pdf"

if PDF_PATH:
    doc  = fitz.open(PDF_PATH)
    text = "\n".join(page.get_text() for page in doc)
    doc.close()
    print(f"Loaded: {len(text):,} characters from {PDF_PATH}")
else:
    print("No PDF set — using built-in demo circular")
    text = """
CIRCULAR NO. 01/2023 — INTEGRATED TAX (RATE)

Section 1: Background
This circular clarifies the GST rate applicable to Information Technology
and IT-Enabled Services (ITES) supplied by Indian entities to domestic
and overseas recipients.

Section 2: Domestic B2B Supply
Domestic B2B supplies of IT/ITES services attract GST at 18 percent,
comprising 9 percent CGST plus 9 percent SGST for intra-state transactions,
or 18 percent IGST for inter-state transactions under Section 5 of the
IGST Act 2017.

Section 3: Conditions for Export of Services
IT services supplied to recipients outside India qualify as export of services
under Section 13 of the IGST Act 2017 when all four conditions are met:
(a) the supplier is located in India,
(b) the recipient is located outside India,
(c) the place of supply is outside India,
(d) payment is received in convertible foreign exchange.

Section 4: Zero-Rating When LUT Is Filed
A registered supplier may export IT services without payment of IGST by
filing a Letter of Undertaking (LUT) in Form GST RFD-11 under Rule 96A of
the CGST Rules 2017. When an LUT is filed, the applicable GST rate is 0
percent (zero-rated). When no LUT is filed, IGST at 18 percent applies
and the supplier may claim a refund under Section 54 of the CGST Act.

Section 5: SaaS and Cloud Services
Software-as-a-Service (SaaS) subscriptions and cloud computing services
provided to foreign entities qualify as export of services when the four
conditions in Section 3 are satisfied. The annual subscription fee invoiced
in foreign currency constitutes the taxable value for the purpose of
calculating the refund of input tax credit.

Section 6: Reverse Charge on Legal Services
Legal services provided by an individual advocate to any business entity
are subject to Reverse Charge Mechanism (RCM) under Section 9(3) of the
CGST Act 2017. The recipient (the registered business) is liable to pay
GST at 18 percent. The advocate does not collect or deposit this tax.

Section 7: Filing and Reporting
Exports of services must be reported in Table 6A of Form GSTR-1. A Foreign
Inward Remittance Certificate (FIRC) or bank advice must be maintained as
documentary evidence that payment was received in convertible foreign exchange.
The GSTR-1 for monthly filers is due by the 11th of the following month.
"""

print(f"Text length: {len(text):,} characters")
print(f"\nFirst 300 characters:\n{text.strip()[:300]}")

No PDF set — using built-in demo circular
Text length: 2,318 characters

First 300 characters:
CIRCULAR NO. 01/2023 — INTEGRATED TAX (RATE)

Section 1: Background
This circular clarifies the GST rate applicable to Information Technology
and IT-Enabled Services (ITES) supplied by Indian entities to domestic
and overseas recipients.

Section 2: Domestic B2B Supply
Domestic B2B supplies of IT/IT


## Step 4 — Split into Chunks

We split the text into 400-word chunks with a 50-word overlap.

**Why overlap?** A sentence at the boundary between two chunks will appear in both,
so it is never lost. Without overlap, a key clause that falls at a boundary
could be split mid-sentence.

In [21]:
CHUNK_SIZE = 80
OVERLAP    = 10

words  = text.split()
step   = CHUNK_SIZE - OVERLAP
chunks = []

for i in range(0, len(words), step):
    chunk_words = words[i : i + CHUNK_SIZE]
    if len(chunk_words) < 30:      # skip tiny trailing fragments
        continue
    chunks.append({
        "id":   len(chunks),
        "text": " ".join(chunk_words),
    })

print(f"Produced {len(chunks)} chunks")
print(f"\nChunk 0 (first 200 characters):\n{chunks[0]['text'][:200]}...")

Produced 5 chunks

Chunk 0 (first 200 characters):
CIRCULAR NO. 01/2023 — INTEGRATED TAX (RATE) Section 1: Background This circular clarifies the GST rate applicable to Information Technology and IT-Enabled Services (ITES) supplied by Indian entities ...


## Step 5 — Embed Every Chunk

We call the Azure embeddings API once per chunk.
Each chunk gets a **vector** — a list of 1536 numbers.

Chunks about similar topics will have similar vectors.
That similarity is what makes search work in Notebook 6B.

In [22]:
print(f"Embedding {len(chunks)} chunks...")

for i, chunk in enumerate(chunks):
    response        = client.embeddings.create(
                          model = AZURE_EMBEDDING_DEPLOYMENT,
                          input = chunk["text"]
                      )
    chunk["vector"] = response.data[0].embedding
    print(f"  [{i+1}/{len(chunks)}] embedded", end="\r")

print(f"\n✅ Done. Each chunk has a vector of {len(chunks[0]['vector'])} numbers.")

Embedding 5 chunks...
  [5/5] embedded
✅ Done. Each chunk has a vector of 1536 numbers.


## Step 6 — Save the Store

We save all chunks (text + vectors) to `store.json`.
Notebook 6B loads this file to run queries — no re-embedding needed.

In [23]:
with open("store.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False)

print(f"✅ Saved {len(chunks)} chunks to store.json")
print(f"   File size: ~{len(json.dumps(chunks))//1024} KB")
print()
print("Open Notebook 6B to ask questions against this store.")

✅ Saved 5 chunks to store.json
   File size: ~170 KB

Open Notebook 6B to ask questions against this store.


---
## What you built

```
PDF  →  text  →  chunks  →  vectors  →  store.json
```

The store is just a list of dicts — each with a `text` and a `vector`.
No database, no server, no external dependencies.

**Notebook 6B** takes a question, embeds it,
finds the closest vectors in `store.json`, and asks GPT to answer from those chunks.